In [2]:
import cv2
import psycopg2
from ultralytics import YOLO
from datetime import datetime
import math
import numpy
conn = psycopg2.connect("dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402")
video_path = 'Модуль В/traffic_full_30min.csv' 

In [ ]:
# Данные БД
DB_URL = "dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402"
CSV_PATH = 'Модуль В/traffic_full_30min.csv'

print(f"[{datetime.now().strftime('%H:%M:%S')}] >>> Скрипт запущен")

# --- 1. ПРОВЕРКА ФАЙЛА И ОБУЧЕНИЕ ---
if not os.path.exists(CSV_PATH):
    print(f"❌ ОШИБКА: Файл {CSV_PATH} не найден! Проверь путь.")
    exit()

try:
    print("🧠 Начинаю чтение CSV и обучение моделей...")
    df = pd.read_csv(CSV_PATH)
    df.columns = ['timestamp', 'intensity_30m', 'avg_speed']
    
    # Расчет "мгновенного" количества машин
    df['moment_cars'] = df['intensity_30m'] / 30.0
    
    # Цели для прогноза
    df['target_30m'] = df['moment_cars'].shift(-1)
    df['target_60m'] = df['moment_cars'].shift(-2)
    df['target_120m'] = df['moment_cars'].shift(-4)
    df_clean = df.dropna()

    X_train = df_clean[['moment_cars']]
    
    model_30 = RandomForestRegressor(n_estimators=30, max_depth=5).fit(X_train, df_clean['target_30m'])
    model_60 = RandomForestRegressor(n_estimators=30, max_depth=5).fit(X_train, df_clean['target_60m'])
    model_120 = RandomForestRegressor(n_estimators=30, max_depth=5).fit(X_train, df_clean['target_120m'])
    
    print("✅ Модели обучены. Перехожу к циклу прогнозирования...")
except Exception as e:
    print(f"❌ Критическая ошибка при подготовке: {e}")
    exit()

# Прогноз
def predict_and_save():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Поиск новых данных в traffic_raw...", end=" ")
    try:
        conn = psycopg2.connect(DB_URL)
        cur = conn.cursor()
        
        # Запрос за ПРЕДЫДУЩУЮ 1 МИНУТУ
        cur.execute("""
            SELECT COUNT(DISTINCT track_id), COALESCE(AVG(speed), 0)
            FROM traffic_raw
            WHERE detection_time >= NOW() - INTERVAL '1 minute'
        """)
        row = cur.fetchone()
        
        # Если данных в базе нет, row[0] будет 0
        curr_count = float(row[0]) if row and row[0] is not None else 0.0
        curr_speed = float(row[1]) if row and row[1] is not None else 0.0
        
        print(f"Найдено: {int(curr_count)} авт.")

        if curr_count >= 0: # Считаем даже если 0, чтобы была динамика в DataLens
            # Прогнозы
            p30 = max(0, int(round(model_30.predict([[curr_count]])[0])))
            p60 = max(0, int(round(model_60.predict([[curr_count]])[0])))
            p120 = max(0, int(round(model_120.predict([[curr_count]])[0])))

            # Сохранение
            now = datetime.now()
            for t_shift, horizon, val in [(30, 30, p30), (60, 60, p60), (120, 120, p120)]:
                cur.execute("""
                    INSERT INTO traffic_predictions (target_time, horizon_minutes, predicted_intensity, predicted_avg_speed)
                    VALUES (%s, %s, %s, %s)
                """, (now + timedelta(minutes=t_shift), horizon, val, curr_speed))
            
            conn.commit()
            print(f"   📈 Прогнозы записаны: +30м={p30}, +60м={p60}, +120м={p120}")
        
        cur.close()
        conn.close()
    except Exception as e:
        print(f"\n❌ Ошибка в цикле: {e}")

# Запуск 
while True:
    predict_and_save()
    time.sleep(15)